<a href="https://colab.research.google.com/github/Hanzet22/GAN-Furry-Art-Photo/blob/main/GAN%20Furry%20V2%20(Art%20And%20Photo%20(NEW)).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HyperGaruda Real-ESRGAN — Fixed Version
**Model tersedia:**
- `anime` — Ilustrasi 2D / Furry Art
- `photo` — Foto Fursuit / Real Photo

Pilih model di cell berikutnya sebelum run.

# HyperGaruda Real-ESRGAN — Fixed Version
**Available models:**
- `anime` — 2D illustrations / Furry art
- `photo` — Fursuit photos / Real photos

Select a model in the next cell before running.

In [ ]:
# ==========================================
# CONFIG — PILIH MODEL DI SINI
# ==========================================
# 'anime' = ilustrasi 2D, furry art, digital art
# 'photo' = foto fursuit, foto real, IRL photo
MODE = 'photo'  # ganti ke 'photo' kalau mau upscale foto fursuit

if MODE == 'anime':
    MODEL_NAME = 'RealESRGAN_x4plus_anime_6B'
    MODEL_URL  = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth'
    MODEL_FILE = 'weights/RealESRGAN_x4plus_anime_6B.pth'
elif MODE == 'photo':
    MODEL_NAME = 'RealESRGAN_x4plus'
    MODEL_URL  = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    MODEL_FILE = 'weights/RealESRGAN_x4plus.pth'
else:
    raise ValueError('MODE harus anime atau photo')

print(f'✅ Mode: {MODE.upper()} | Model: {MODEL_NAME}')

In [ ]:
import os

# ==========================================
# 1. CLONE REPO
# ==========================================
print('📥 Cloning Real-ESRGAN...')
if not os.path.exists('Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git
print('✅ Done.')

if not os.path.exists('inference_realesrgan.py'):
    %cd Real-ESRGAN
    print('✅ Entered Real-ESRGAN folder.')
else:
    print('✅ Already in Real-ESRGAN folder.')

# ==========================================
# 2. INSTALL DEPENDENCIES
# ==========================================
print('🔧 Installing dependencies...')
!pip install -q basicsr facexlib gfpgan
!pip install -q -r requirements.txt
!python setup.py develop -q
print('✅ Dependencies installed.')

# ==========================================
# 3. FIX BASICSR COMPATIBILITY
# torchvision baru hapus functional_tensor
# basicsr 1.4.2 masih pake yang lama -> patch manual
# ==========================================
print('🔧 Patching basicsr compatibility...')
import site

files_to_patch = [
    '/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py',
    '/usr/local/lib/python3.12/dist-packages/basicsr/utils/img_process_util.py',
]

for fpath in files_to_patch:
    if not os.path.exists(fpath):
        print(f'⚠️ Skip (not found): {fpath}')
        continue
    with open(fpath, 'r') as f:
        content = f.read()
    patched = content.replace(
        'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
        'from torchvision.transforms.functional import rgb_to_grayscale'
    )
    with open(fpath, 'w') as f:
        f.write(patched)
    if patched != content:
        print(f'✅ Patched: {os.path.basename(fpath)}')
    else:
        print(f'✅ No patch needed: {os.path.basename(fpath)}')

print('✅ Patch done.')

In [ ]:
import os

# ==========================================
# 4. DOWNLOAD MODEL
# ==========================================
os.makedirs('weights', exist_ok=True)

if not os.path.exists(MODEL_FILE):
    print(f'⏳ Downloading {MODEL_NAME}...')
    !wget -q {MODEL_URL} -P weights/
    print('✅ Model downloaded.')
else:
    print('✅ Model already exists.')

# ==========================================
# 5. UPLOAD GAMBAR
# ==========================================
from google.colab import files
import glob

os.makedirs('inputs', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f'📂 Upload gambar ({MODE} mode)...')
uploaded = files.upload()

for filename in uploaded.keys():
    new_name = filename.replace(' ', '_')
    !mv "{filename}" "inputs/{new_name}"
    print(f'   ➕ {new_name}')

In [ ]:
# ==========================================
# 6. UPSCALE
# ==========================================
print(f'🚀 Upscaling ({MODE.upper()} mode | {MODEL_NAME})...')

!python inference_realesrgan.py \
    --model_path {MODEL_FILE} \
    --model_name {MODEL_NAME} \
    --input inputs \
    --output results \
    -s 4 \
    --ext png \
    --tile 512 \
    --tile_pad 32

In [ ]:
# ==========================================
# 7. DOWNLOAD HASIL
# ==========================================
from google.colab import files
import glob

print('\n🎉 Done! Preparing download...')

if os.path.exists('results') and os.listdir('results'):
    result_files = glob.glob('results/*.png')
    print(f'✅ {len(result_files)} image(s) upscaled.')
    !zip -r /content/results.zip results
    files.download('/content/results.zip')
    print('📥 Download started.')
else:
    print('⚠️ No results found. Check error logs above.')